# P7-EXPAND-GEO-SITE
**Task:** Expand `geographic_site` from 300 → 2,969 rows to resolve all FK orphans in `service_inventory.site_a_id` / `site_z_id`.

**Orphan breakdown:** 2,669 missing site IDs (1,900 CP + 384 CO + 385 DC) across 10 states.

**Approach:**
1. Extract all orphan site IDs from service_inventory
2. Parse state code from site_id pattern (e.g., `CO-AZ-001` → state=AZ)
3. Map state → region/city/lat/lon from existing geographic_site patterns
4. Generate realistic coordinates with jitter around known city centers
5. INSERT INTO geographic_site with Lakelink Fiber branding
6. Validate FK integrity post-expansion

**Catalog/Schema:** `cdm_tmforum.provable_resilience_intelligence`

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
import hashlib

SCHEMA = "cdm_tmforum.provable_resilience_intelligence"

# ── Load current geographic_site ──
geo_site = spark.table(f"{SCHEMA}.geographic_site")
print(f"Current geographic_site rows: {geo_site.count()}")

# ── Find all distinct site IDs referenced by service_inventory ──
si = spark.table(f"{SCHEMA}.service_inventory")
si_site_ids = (
    si.select(F.col("site_a_id").alias("site_id"))
    .union(si.select(F.col("site_z_id").alias("site_id")))
    .distinct()
)

# ── Orphan site IDs = referenced but not in geographic_site ──
orphans = si_site_ids.join(geo_site.select("site_id"), "site_id", "left_anti")
orphan_count = orphans.count()
print(f"Orphan site IDs to generate: {orphan_count}")

# ── Parse state code and site type from ID pattern ──
# Pattern: CP-AZ-001, CO-NY-023, DC-GA-005
orphans_parsed = orphans.withColumn(
    "prefix", F.split("site_id", "-")[0]
).withColumn(
    "state", F.split("site_id", "-")[1]
).withColumn(
    "site_type",
    F.when(F.col("prefix") == "CP", "customer_premise")
     .when(F.col("prefix") == "CO", "central_office")
     .when(F.col("prefix") == "DC", "data_center")
     .otherwise("unknown")
)

orphans_parsed.groupBy("site_type").count().orderBy("site_type").show()

# ── Build state → region lookup from existing data ──
state_region = geo_site.select("state", "region").distinct()
print("State-to-region mapping:")
state_region.orderBy("state").show(50, truncate=False)

In [0]:
# ── Build state → (city, center_lat, center_lon) reference from existing sites ──
# For each state, compute the centroid of existing sites as the city anchor
state_centroids = (
    geo_site
    .groupBy("state", "region")
    .agg(
        F.avg("latitude").alias("center_lat"),
        F.avg("longitude").alias("center_lon"),
        F.first("city").alias("primary_city"),
        F.stddev("latitude").alias("lat_stddev"),
        F.stddev("longitude").alias("lon_stddev"),
        F.count("*").alias("existing_count")
    )
    .withColumn("lat_stddev", F.coalesce(F.col("lat_stddev"), F.lit(0.05)))
    .withColumn("lon_stddev", F.coalesce(F.col("lon_stddev"), F.lit(0.05)))
)

print("State centroids for site generation:")
state_centroids.orderBy("state").show(50, truncate=False)

In [0]:
import random
from datetime import datetime

# ── Collect reference data ──
centroid_map = {}
for row in state_centroids.collect():
    centroid_map[row["state"]] = {
        "region": row["region"],
        "center_lat": row["center_lat"],
        "center_lon": row["center_lon"],
        "primary_city": row["primary_city"],
        "lat_stddev": row["lat_stddev"],
        "lon_stddev": row["lon_stddev"]
    }

# Secondary cities per state for customer_premise diversity
STATE_CITIES = {
    "AZ": ["Phoenix", "Tucson", "Mesa", "Scottsdale", "Chandler"],
    "CO": ["Denver", "Colorado Springs", "Aurora", "Boulder", "Fort Collins"],
    "FL": ["Jacksonville", "Tampa", "Miami", "Orlando", "St. Petersburg"],
    "GA": ["Atlanta", "Savannah", "Augusta", "Columbus", "Macon"],
    "MN": ["Minneapolis", "St. Paul", "Rochester", "Duluth", "Bloomington"],
    "NC": ["Raleigh", "Charlotte", "Durham", "Greensboro", "Winston-Salem"],
    "NY": ["New York", "Buffalo", "Rochester", "Syracuse", "Albany"],
    "OH": ["Columbus", "Cleveland", "Cincinnati", "Toledo", "Akron"],
    "TX": ["Dallas", "Houston", "Austin", "San Antonio", "Fort Worth"],
    "WA": ["Seattle", "Spokane", "Tacoma", "Vancouver", "Bellevue"],
    "AL": ["Birmingham", "Montgomery", "Huntsville"],
    "AR": ["Little Rock", "Fayetteville"],
    "CT": ["Hartford", "New Haven"],
    "DC": ["Washington"],
    "IA": ["Des Moines", "Cedar Rapids"],
    "ID": ["Boise", "Meridian"],
    "IN": ["Indianapolis", "Fort Wayne"],
    "KS": ["Wichita", "Overland Park"],
    "KY": ["Louisville", "Lexington"],
    "LA": ["New Orleans", "Baton Rouge"],
    "MA": ["Boston", "Worcester"],
    "MI": ["Detroit", "Grand Rapids"],
    "MO": ["Kansas City", "St. Louis"],
    "NE": ["Omaha", "Lincoln"],
    "NM": ["Albuquerque", "Santa Fe"],
    "NV": ["Las Vegas", "Reno"],
    "OK": ["Oklahoma City", "Tulsa"],
    "OR": ["Portland", "Eugene"],
    "PA": ["Philadelphia", "Pittsburgh"],
    "TN": ["Nashville", "Memphis"],
    "UT": ["Salt Lake City", "Provo"],
    "VA": ["Richmond", "Virginia Beach"],
    "WI": ["Milwaukee", "Madison"],
}

STREET_NAMES = ["Main St", "Elm St", "Oak Ave", "Industrial Blvd", "Commerce Dr",
                "Tech Pkwy", "Enterprise Way", "Network Rd", "Fiber Ln", "Campus Dr",
                "Park Ave", "Market St", "Broadway", "First Ave", "Pine St",
                "Maple Dr", "Cedar Blvd", "Summit Rd", "Valley Way", "Ridge Pkwy"]

CLLI_CITY = {
    "AZ": "PHAZ", "CO": "DNVR", "FL": "JCVL", "GA": "ATLG", "MN": "MPLS",
    "NC": "RLGH", "NY": "NYCM", "OH": "CLMB", "TX": "DLLS", "WA": "STLN",
    "AL": "BRHM", "AR": "LTLR", "CT": "HTFD", "DC": "WASH", "IA": "DSMN",
    "ID": "BOIS", "IN": "INDP", "KS": "WCHT", "KY": "LSVL", "LA": "NWOR",
    "MA": "BSTN", "MI": "DTRT", "MO": "KCMO", "NE": "OMAH", "NM": "ALBQ",
    "NV": "LSVG", "OK": "OKCM", "OR": "PRTL", "PA": "PHLA", "TN": "NSHL",
    "UT": "SLKC", "VA": "RCMD", "WI": "MLWK"
}

# Fallback for states not in existing data
STATE_FALLBACK = {
    "AZ": {"region":"West","center_lat":33.45,"center_lon":-112.07,"primary_city":"Phoenix","lat_stddev":0.5,"lon_stddev":0.5},
    "CO": {"region":"West","center_lat":39.74,"center_lon":-104.99,"primary_city":"Denver","lat_stddev":0.1,"lon_stddev":0.1},
    "FL": {"region":"Southeast","center_lat":30.33,"center_lon":-81.66,"primary_city":"Jacksonville","lat_stddev":0.3,"lon_stddev":0.3},
    "GA": {"region":"Southeast","center_lat":33.75,"center_lon":-84.39,"primary_city":"Atlanta","lat_stddev":0.2,"lon_stddev":0.2},
    "MN": {"region":"Central","center_lat":44.98,"center_lon":-93.27,"primary_city":"Minneapolis","lat_stddev":0.1,"lon_stddev":0.1},
    "NC": {"region":"Southeast","center_lat":35.78,"center_lon":-78.64,"primary_city":"Raleigh","lat_stddev":0.2,"lon_stddev":0.2},
    "NY": {"region":"East","center_lat":40.71,"center_lon":-74.01,"primary_city":"New York","lat_stddev":0.2,"lon_stddev":0.2},
    "OH": {"region":"East","center_lat":39.96,"center_lon":-82.99,"primary_city":"Columbus","lat_stddev":0.2,"lon_stddev":0.2},
    "TX": {"region":"Central","center_lat":32.78,"center_lon":-96.80,"primary_city":"Dallas","lat_stddev":0.5,"lon_stddev":0.5},
    "WA": {"region":"West","center_lat":47.61,"center_lon":-122.33,"primary_city":"Seattle","lat_stddev":0.2,"lon_stddev":0.2},
}

def generate_site(site_id, site_type, state, seq, rng):
    ref = centroid_map.get(state, STATE_FALLBACK.get(state, {
        "region": "Central", "center_lat": 39.0, "center_lon": -95.0,
        "primary_city": "Unknown", "lat_stddev": 0.5, "lon_stddev": 0.5
    }))
    spread = 1.5 if site_type == "customer_premise" else 0.8
    lat = ref["center_lat"] + rng.gauss(0, ref["lat_stddev"] * spread)
    lon = ref["center_lon"] + rng.gauss(0, ref["lon_stddev"] * spread)
    lat = max(24.5, min(49.0, lat))
    lon = max(-125.0, min(-66.9, lon))
    cities = STATE_CITIES.get(state, [ref.get("primary_city", "Unknown")])
    city = rng.choice(cities)
    type_labels = {"customer_premise": "CP", "central_office": "CO", "data_center": "DC"}
    site_name = f"Lakelink Fiber {city} {type_labels[site_type]}-{seq}"
    city_code = CLLI_CITY.get(state, state[:2].upper() + "XX")
    tc = {"customer_premise": "CP", "central_office": "CO", "data_center": "DC"}[site_type]
    clli = f"{city_code}{tc}{seq % 100:02d}"
    tier = "tier_3" if site_type == "customer_premise" else rng.choice(["tier_1", "tier_1", "tier_2"])
    return {
        "site_id": site_id, "site_name": site_name, "site_type": site_type,
        "clli_code": clli, "latitude": round(lat, 12), "longitude": round(lon, 12),
        "address": f"{rng.randint(100,9999)} {rng.choice(STREET_NAMES)}",
        "city": city, "state": state, "region": ref["region"], "tier": tier,
        "is_active": True, "created_at": datetime(2026, 10, 21, 0, 0, 0)
    }

# ── Generate all orphan sites ──
rng = random.Random(42)
orphan_rows = orphans_parsed.collect()
new_sites = []
for row in orphan_rows:
    sid, stype, state = row["site_id"], row["site_type"], row["state"]
    try:
        seq = int(sid.split("-")[-1])
    except:
        seq = rng.randint(1, 999)
    new_sites.append(generate_site(sid, stype, state, seq, rng))

print(f"Generated {len(new_sites)} new site records")
from collections import Counter
type_counts = Counter(s['site_type'] for s in new_sites)
print(f"By type: {dict(sorted(type_counts.items()))}")
region_counts = Counter(s['region'] for s in new_sites)
print(f"By region: {dict(sorted(region_counts.items()))}")